# PPG Agent Evaluation on Test Data

This notebook evaluates a pre‑trained **Phasic Policy Gradient (PPG)** agent on out‑of‑sample financial data.  
The PPG implementation follows Cobbe et al. (ICML 2021) and is optimised for discrete trading actions (‑1, 0, +1).

We will:

1. Load the test dataset and the saved checkpoint.
2. Automatically infer the network architecture from the checkpoint.
3. Recreate the trading environment with the same feature set used during training.
4. Run a deterministic (or stochastic) evaluation rollout.
5. Compute performance metrics and plot the equity curve and drawdown.

---



In [ ]:
# PPG Evaluation on Test Data

import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Add project root to path
sys.path.append('..')
from environment.trading_env import TradingEnv
from experts.ppg_expert import PPGAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')



---

## 1. Configuration

Set the asset symbol, paths to the configuration file, test data, and the saved model checkpoint.  
The hyperparameters are read from `experts.yaml` (under the `ppg` key).



In [ ]:
ASSET = "BTCUSDT"
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/train.csv"
MODEL_PATH = f"../checkpoints/{ASSET}/no_diversity/PPG/ppg_expert.pt"

# Load configuration
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)
ppg_cfg = cfg["ppg"]   # PPG‑specific hyperparameters



---

## 2. Load Test Data & Checkpoint

Read the test data and load the model checkpoint.  
The checkpoint contains the network state dictionary, but may not store feature names or environment dimensions – we will derive them.



In [ ]:
# Load test data
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"Test data shape: {test_df.shape}")

# Load checkpoint
ckpt = torch.load(MODEL_PATH, map_location='cpu')

# Extract state dict (handle different save formats)
if "network" in ckpt:
    state_dict = ckpt["network"]
    saved_features = ckpt.get("features", None)
    saved_n_state = ckpt.get("n_state_features", None)
else:
    state_dict = ckpt
    saved_features = None
    saved_n_state = None

print("Checkpoint loaded.")



---

## 3. Infer Network Architecture from the State Dict

The `PPGAgent` uses a `SharedActorCritic` network with a shared encoder (`shared_encoder`) and separate policy (`actor_head`) and value (`critic_head`) heads.  
We inspect the keys of the state dictionary to determine:

- `state_dim` – input dimension (from the first linear layer)
- `hidden_dim` – number of units per hidden layer
- `num_layers` – number of linear layers in the shared encoder
- `action_dim` – number of discrete actions (from `actor_head.weight`)



In [ ]:
# Find all shared_encoder linear layer weights
encoder_weight_keys = [k for k in state_dict.keys() 
                       if k.startswith("shared_encoder.") and k.endswith(".weight")]

if not encoder_weight_keys:
    raise KeyError("No 'shared_encoder.*.weight' keys found. Is this a PPGAgent checkpoint?")

# Sort by layer index (0, 2, 4, ...)
encoder_weight_keys.sort(key=lambda x: int(x.split('.')[1]))

# Determine number of linear layers (each layer has a weight)
num_layers = len(encoder_weight_keys)

# Infer hidden_dim from output size of the first linear layer
first_weight = state_dict[encoder_weight_keys[0]]
hidden_dim = first_weight.shape[0]   # output dimension

# State dimension from input size of the first layer
state_dim_from_weights = first_weight.shape[1]

# Action dimension from actor_head
if "actor_head.weight" in state_dict:
    action_dim = state_dict["actor_head.weight"].shape[0]
else:
    action_dim = 3   # fallback

# Optionally infer activation – not stored, so we default to 'tanh'
activation = "tanh"

print(f"Inferred architecture: hidden_dim={hidden_dim}, num_layers={num_layers}, "
      f"action_dim={action_dim}, state_dim={state_dim_from_weights}")



---

## 4. Determine Feature Columns and Environment State Dimension

The environment needs a list of feature columns and the number of internal state features (account balance, position, etc.).  
We try to use the features from the checkpoint or from the config; if not available, we auto‑select from the test data.



In [ ]:
# Number of environment‑internal state features (balance, position, etc.)
# This is defined in TradingEnv._get_state() – typically 6 (equity, position, etc.)
# If not saved, we assume the standard value.
n_state_features = saved_n_state if saved_n_state is not None else 6

# Expected number of feature columns = state_dim_from_weights - n_state_features
expected_features = state_dim_from_weights - n_state_features
print(f"Environment state features (assumed): {n_state_features}")
print(f"Checkpoint expects {expected_features} feature columns.")

# Select feature columns
if saved_features is not None and len(saved_features) == expected_features:
    features = saved_features
    print("Using features from checkpoint:", features)
else:
    config_features = ppg_cfg.get("features", [])
    if len(config_features) == expected_features:
        features = config_features
        print("Using features from config:", features)
    else:
        # Auto‑select from test_df columns, excluding price/volume base columns
        exclude = ["open", "high", "low", "close", "volume", "log_volume", "returns", "return_5"]
        candidates = [c for c in test_df.columns if c not in exclude]
        if len(candidates) >= expected_features:
            features = candidates[:expected_features]
        else:
            features = test_df.columns[:expected_features].tolist()
        print("Auto‑selected features:", features)

# Ensure the computed state_dim matches what the network expects
state_dim = len(features) + n_state_features
if state_dim != state_dim_from_weights:
    print(f"WARNING: Computed state_dim ({state_dim}) does not match expected ({state_dim_from_weights}).")
    print("You may need to adjust n_state_features or select different features.")



---

## 5. Create Trading Environment

Instantiate the `TradingEnv` with the selected features and the parameters from the configuration.



In [ ]:
asset_type = "crypto" if "USDT" in ASSET else "forex"
window_size = cfg.get("window_size", 20)

env_params = {
    "df": test_df,
    "feature_cols": features,
    "expert_id": 6,                     # PPG expert id
    "asset_type": asset_type,
    "contract_size": 1.0 if asset_type == "crypto" else 100000.0,
    "commission_rate": 0.0005 if asset_type == "crypto" else 1.5/100000.0,
    "spread_rate": 0.0002 if asset_type == "crypto" else 0.00005,
    "slippage_rate": 0.0001 if asset_type == "crypto" else 0.00001,
    "leverage": 5 if asset_type == "crypto" else 30,
    "maintenance_margin_ratio": 0.5,
    "initial_balance": 10000.0,
    "window_size": window_size,
    "reward_scale": 1.0,
    "clip_reward": (-10.0, 10.0),
    "max_position_fraction": 0.95,
    "continuous_actions": False,
}

env = TradingEnv(**env_params)
print("Environment created.")



---

## 6. Instantiate the PPG Agent

Create a `PPGAgent` with the inferred architecture.  
We set the training‑related hyperparameters to dummy values (they are not used during inference).  
**Important:** The notebook previously included `policy_kl_range` – this is **not** a parameter of `PPGAgent`; we remove it and optionally set `target_kl`.



In [ ]:
# Instantiate agent with inferred dimensions
agent = PPGAgent(
    state_dim=state_dim,          # must match state_dim_from_weights
    action_dim=action_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    activation=activation,
    # The following are not used during inference but required by __init__
    lr=1e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    policy_epochs=10,
    batch_size=256,
    rollout_steps=2048,
    aux_epochs=6,
    aux_batch_size=256,
    kl_coef=0.1,
    target_kl=0.02,          # optional, but valid
    # Do NOT use policy_kl_range
)

# Load the network state dictionary
agent.network.load_state_dict(state_dict)
print("Model loaded successfully. Ready for inference.")



---

## 7. Run Evaluation Rollout

Reset the environment and step through all time steps, recording the equity after each step.  
The agent’s `act()` method returns `(action, log_prob, value)` – we only need the action.



In [ ]:
obs, _ = env.reset()
done = False
equity_curve = [env.initial_balance]

while not done:
    action, _, _ = agent.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    equity_curve.append(info.get('equity', env.initial_balance))
    done = terminated or truncated

equity_curve = np.array(equity_curve)
cum_return = (equity_curve - equity_curve[0]) / equity_curve[0]

print(f"Final equity: {equity_curve[-1]:.2f}")
print(f"Final cumulative return: {cum_return[-1]:.4f} ({cum_return[-1]*100:.2f}%)")



---

## 8. Performance Metrics

Compute key trading statistics:

- Total return
- Annualised return and volatility (assuming 252 trading days)
- Sharpe ratio
- Maximum drawdown
- Win rate



In [ ]:
daily_returns = np.diff(equity_curve) / equity_curve[:-1]
total_ret = cum_return[-1]
n = len(daily_returns)
ann_ret = (1 + total_ret) ** (252 / n) - 1 if n > 0 else 0.0
ann_vol = np.std(daily_returns) * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else 0.0

# Drawdown
peak = np.maximum.accumulate(equity_curve)
drawdown = (peak - equity_curve) / peak
max_dd = np.max(drawdown)

win_rate = np.mean(daily_returns > 0)

print("\n--- PPG Performance ---")
print(f"Total Return:       {total_ret:.2%}")
print(f"Annualized Return:  {ann_ret:.2%}")
print(f"Annualized Vol:     {ann_vol:.2%}")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Max Drawdown:       {max_dd:.2%}")
print(f"Win Rate:           {win_rate:.2%}")



---

## 9. Visualisations

Plot the equity curve (cumulative return) and the drawdown over time.



In [ ]:
# Equity curve (cumulative return)
plt.figure(figsize=(14, 6))
plt.plot(cum_return, label='PPG', linewidth=2)
plt.title("PPG Equity Curve (Cumulative Return)")
plt.xlabel("Step")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.legend()
plt.show()

# Drawdown
plt.figure(figsize=(14, 3))
plt.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
plt.plot(drawdown, color='darkred', linewidth=0.5)
plt.title("Drawdown")
plt.xlabel("Step")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()



---